In [ ]:
from py2neo import Node, Relationship, Graph, NodeMatcher, RelationshipMatcher, Subgraph
g = Graph('bolt://localhost:7687',auth=('neo4j','your password'), name = 'trkg')

In [23]:
import pandas as pd
from tqdm import tqdm
from pathlib import Path

# ---------- locate project root ----------
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "Main data").exists():
            return parent
    raise RuntimeError("Project root (CODE UPLOAD) not found.")

PROJECT_ROOT = find_project_root()

# 1. 读取 CSV 文件并提取列名（跳过第一列）
csv_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "completed_tensor_filtered.csv"
)

df = pd.read_csv(csv_path)
column_names = df.columns.tolist()[1:]

print(len(column_names))
print(column_names)


265
['92973', '92974', '92986', '92989', '92992', '92994', '92995', '92996', '93001', '93002', '93003', '93016', '93017', '93022', '93592', '93597', '93600', '93601', '93621', '93625', '93627', '93629', '93630', '93631', '93632', '93653', '93665', '93666', '93667', '93669', '93671', '93676', '93682', '93684', '93697', '93699', '93700', '93702', '93703', '93707', '93708', '93709', '93711', '93712', '93713', '93714', '93716', '93719', '93723', '93724', '93725', '93727', '93728', '93729', '93730', '93734', '93839', '93840', '93842', '93844', '93845', '93848', '93852', '93855', '93859', '93862', '93863', '93870', '93871', '93872', '93873', '93876', '93885', '93891', '93893', '93897', '93898', '93899', '93901', '93904', '93908', '93909', '93910', '93911', '93912', '93915', '93916', '93918', '93919', '93922', '93924', '93925', '93927', '93932', '93933', '93935', '93936', '93937', '93938', '93939', '93940', '93942', '93943', '93944', '93945', '93948', '93954', '93955', '93958', '93960', '9396

In [24]:
a = ', '.join(str(name) for name in column_names)
a

'92973, 92974, 92986, 92989, 92992, 92994, 92995, 92996, 93001, 93002, 93003, 93016, 93017, 93022, 93592, 93597, 93600, 93601, 93621, 93625, 93627, 93629, 93630, 93631, 93632, 93653, 93665, 93666, 93667, 93669, 93671, 93676, 93682, 93684, 93697, 93699, 93700, 93702, 93703, 93707, 93708, 93709, 93711, 93712, 93713, 93714, 93716, 93719, 93723, 93724, 93725, 93727, 93728, 93729, 93730, 93734, 93839, 93840, 93842, 93844, 93845, 93848, 93852, 93855, 93859, 93862, 93863, 93870, 93871, 93872, 93873, 93876, 93885, 93891, 93893, 93897, 93898, 93899, 93901, 93904, 93908, 93909, 93910, 93911, 93912, 93915, 93916, 93918, 93919, 93922, 93924, 93925, 93927, 93932, 93933, 93935, 93936, 93937, 93938, 93939, 93940, 93942, 93943, 93944, 93945, 93948, 93954, 93955, 93958, 93960, 93961, 93962, 93965, 93967, 93969, 93971, 93972, 93974, 93975, 93977, 93979, 93980, 93981, 93983, 93984, 93987, 93988, 93989, 93992, 93993, 93994, 93996, 93998, 93999, 94001, 94002, 94005, 94008, 94009, 94011, 94012, 94013, 94015

In [25]:
# 批量参数
batch_size = 10  # 可根据内存和Neo4j响应能力调节
results = []

for i in tqdm(range(0, len(column_names), batch_size), desc="Querying honeycombs"):
    batch = column_names[i:i + batch_size]
    batch_str = ', '.join(str(name) for name in batch)
    
    # if i >= batch_size:
    #     break
    # print(batch)
    
    query = f"""
    MATCH (h:honeycomb)-[:contains]->(p:POI)
    WHERE h.name IN [{batch_str}]
      AND p.type_main IN [
        '交通设施服务', '住宿服务', '公司企业',
        '医疗保健服务', '购物服务',
        '风景名胜', '餐饮服务'
      ]
    RETURN h.name AS honeycomb_name, p.type_main AS type_main, count(p) AS count
    ORDER BY honeycomb_name, count DESC
    """
    
    df_batch = g.run(query).to_data_frame()
    results.append(df_batch)

# 4. 合并并输出结果
final_result = pd.concat(results, ignore_index=True)
# final_result

Querying honeycombs: 100%|████████████████████████████████████████████████████████████| 27/27 [00:00<00:00, 203.39it/s]


In [26]:
final_result

,honeycomb_name,type_main,count
0,92973,购物服务,9
1,92973,餐饮服务,4
2,92973,交通设施服务,1
3,92974,公司企业,4
4,92974,交通设施服务,3
...,...,...,...
1413,95476,住宿服务,19
1414,95476,交通设施服务,11
1415,95476,公司企业,6
1416,95476,餐饮服务,5


In [27]:
# 1. 所有 honeycomb 列表，转换为 set
all_honeycombs = set(int(x) for x in column_names)

# 2. 已有 POI 的 honeycomb（final_result 中）
honeycombs_with_poi = set(final_result['honeycomb_name'].unique())

# 3. 找出没有 POI 的 honeycomb
honeycombs_without_poi = sorted(all_honeycombs - honeycombs_with_poi)

# 4. 获取所有 type_main（列名）
target_types = sorted(final_result['type_main'].dropna().unique())

# 5. 构造没有 POI 的 honeycomb 对应的 0 值 DataFrame
zero_rows = []
for h in honeycombs_without_poi:
    for t in target_types:
        zero_rows.append({'honeycomb_name': h, 'type_main': t, 'count': 0})

df_zeros = pd.DataFrame(zero_rows)

# 6. 合并原始结果与零填充部分
final_result_filled = pd.concat([final_result, df_zeros], ignore_index=True)

# 7. 生成透视表（honeycomb_name × type_main）
pivot = final_result_filled.pivot_table(
    index='honeycomb_name',
    columns='type_main',
    values='count',
    fill_value=0
)

# 8. 保证所有 type_main 列都在（可选步骤）
pivot = pivot[target_types]
pivot.reset_index(inplace=True)

In [28]:
# ---------- relative output path ----------
output_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "poi_type_count.csv"
)

pivot.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ 结果已保存为 {output_path}")


✅ 结果已保存为 C:\DCLASS\myJupyter\paper2\shanghai_data_save\local_grid_time-series\poi_type_count.csv


In [29]:
len(honeycombs_without_poi)

1

In [30]:
import pandas as pd
import os

# ---------- INPUT: poi_type_count.csv (relative) ----------
file_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "poi_type_count.csv"
)
df = pd.read_csv(file_path)

# 提取所有 POI 类型列（跳过 'honeycomb_name'）
column_names = df.columns.tolist()[1:]

# ---------- OUTPUT: POI folder under prediction_data (relative) ----------
output_dir = PROJECT_ROOT / "Main data" / "prediction_data" / "POI"
output_dir.mkdir(parents=True, exist_ok=True)

# 拆分并保存为横向格式
for i, poi_col in enumerate(column_names, start=1):
    df_sub = df[['honeycomb_name', poi_col]].copy()
    df_transposed = df_sub.set_index('honeycomb_name').T
    df_transposed.index = ['value']

    output_path = output_dir / f"poi{i}.csv"
    df_transposed.to_csv(output_path, encoding='utf-8-sig', index=False, header=False)
    print(f"✅ 已保存横向 CSV: {output_path}")

✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi1.csv
✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi2.csv
✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi3.csv
✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi4.csv
✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi5.csv
✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi6.csv
✅ 已保存横向 CSV: C:\DCLASS\myJupyter\paper2\application\prediction\model\data\POI\poi7.csv
